## Para este modelo vamos usar o pyThorch


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('../data/interim/dataset_10500.csv')

# 1. Criar um codificador para os sintomas (unificando todos os sintomas possíveis)
todos_sintomas = pd.concat([df['Sintoma 1'], df['Sintoma 2']]).unique()
encoder_sintomas = LabelEncoder().fit(todos_sintomas)

# 2. Criar um codificador para as doenças
encoder_doencas = LabelEncoder()
df['target'] = encoder_doencas.fit_transform(df['Doença Associada'])


# Aplicamos o mesmo mapeamento numérico para as duas colunas de sintomas
df['S1_num'] = encoder_sintomas.transform(df['Sintoma 1'])
df['S2_num'] = encoder_sintomas.transform(df['Sintoma 2'])

X = df[['S1_num', 'S2_num']].values
y = df['target'].values

# Converter para Tensores do PyTorch
X_tensor = torch.tensor(X, dtype=torch.long)
y_tensor = torch.tensor(y, dtype=torch.long)

# Define model
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(encoder_sintomas.classes_), 8)
        self.hidden = nn.Linear(16, 64)
        self.hidden2 = nn.Linear(64, 32)
        self.output = nn.Linear(32, len(encoder_doencas.classes_))
    
    def forward(self, x):
        x = self.embedding(x)
        x = x.view(x.size(0), -1) 
        x = torch.relu(self.hidden(x))
        x = torch.relu(self.hidden2(x))
        x = self.output(x)
        return x


model = Net()
optimizer = optim.Adam(model.parameters(), lr=0.002)
criterion = nn.CrossEntropyLoss()

# Training loop
for epoch in range(5000):
    model.train()
    y_pred = model(X_tensor)
    loss = criterion(y_pred, y_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# Vizualizando resultados


In [ ]:
import torch

with open('../data/interim/sintomas_pacientes.txt', encoding='utf-8') as f:
    relatos = f.read().splitlines()

model.eval()

for relato in relatos:
    # Ignorar linhas vazias do arquivo
    if not relato.strip():
        continue
        
    relato_min = relato.lower()
    sintomas_encontrados = [s for s in encoder_sintomas.classes_ if s.lower() in relato_min]
    
    if len(sintomas_encontrados) >= 1:
        if len(sintomas_encontrados) == 1:
            s1 = sintomas_encontrados[0]
            s2 = sintomas_encontrados[0]
        else:
            s1 = sintomas_encontrados[0]
            s2 = sintomas_encontrados[1]

        s1_num = encoder_sintomas.transform([s1])[0]
        s2_num = encoder_sintomas.transform([s2])[0]

        entrada = torch.tensor([[s1_num, s2_num]], dtype=torch.long)
        with torch.no_grad():
            output = model(entrada)
            indice_doenca = torch.max(output, 1)[1]

        doenca_predita = encoder_doencas.inverse_transform(indice_doenca.cpu().numpy())[0]
        print(f"Relato {relato} ->\n Doença Predita: {doenca_predita}")
    else:
        print(f"Relato {relato} ->\n Doença Predita: Não encontrou sintomas.")


Relato 1. [Idade: 45 | Sexo: M | Pressão: Alta | Histórico: Estresse] Há três dias estou com dor de cabeça constante, que piora à noite e tem dificultado minha concentração no trabalho. ->
 Doença Predita: Doenças Cardiovasculares
Relato 2. [Idade: 62 | Sexo: F | Pressão: Normal | Histórico: Diabetes] Sinto cansaço excessivo há uma semana, mesmo dormindo bem, o que tem afetado minha produtividade diária. ->
 Doença Predita: Angina
Relato 3. [Idade: 55 | Sexo: M | Pressão: Alta | Histórico: Tabagista e Colesterol Alto] Desde ontem estou com dor no peito ao fazer esforço físico, o que me impede de realizar minhas atividades normais. ->
 Doença Predita: Infarto Agudo do Miocárdio
Relato 4. [Idade: 78 | Sexo: F | Pressão: Baixa | Histórico: Uso de anti-hipertensivos] Tenho sentido tontura há dois dias, principalmente ao me levantar, o que tem me deixado inseguro para sair de casa. ->
 Doença Predita: Insuficiência Cardíaca
Relato 5. [Idade: 34 | Sexo: M | Pressão: Normal | Histórico: Nenhu